In [11]:
import pandas as pd

df = pd.read_parquet("../data/processed/clean_sales.parquet")
df["week"] = df["InvoiceDate"].dt.to_period("W-SUN").dt.start_time


g = df.groupby("StockCode")["Quantity"]
cap = pd.concat([g.quantile(0.99), g.median() * 10], axis=1).max(axis=1)
df["qty_capped"] = df["Quantity"].clip(upper=df["StockCode"].map(cap))
print("Order lines capped:", (df["Quantity"] > df["qty_capped"]).sum(),
      "of", len(df))

weekly = (df.groupby(["StockCode", "week"])
            .agg(units=("Quantity", "sum"),
                 units_regular=("qty_capped", "sum"),
                 revenue=("Revenue", "sum"),
                 orders=("InvoiceNo", "nunique"),
                 avg_price=("UnitPrice", "mean"))
            .reset_index())
print(weekly.shape, weekly["week"].nunique())

Order lines capped: 3592 of 495525
(96983, 7) 51


In [12]:
stats = (weekly.groupby("StockCode")
               .agg(total_units=("units", "sum"),
                    total_orders=("orders", "sum"),
                    active_weeks=("week", "nunique"),
                    max_week=("units", "max"),
                    median_week=("units", "median"))
               .reset_index())
stats["spike_ratio"] = stats["max_week"] / stats["median_week"]

candidates = stats[stats["active_weeks"] >= 40]
top_by_orders = candidates.nlargest(60, "total_orders")["StockCode"]
top_by_units = candidates.nlargest(60, "total_units")["StockCode"]

print("Overlap between the two rankings:", len(set(top_by_orders) & set(top_by_units)), "of 60")
print(candidates.loc[candidates["StockCode"].isin(top_by_units), "spike_ratio"].describe())
print(candidates.loc[candidates["StockCode"].isin(top_by_orders), "spike_ratio"].describe())

top = top_by_orders

Overlap between the two rankings: 38 of 60
count    60.000000
mean      7.332869
std       9.179009
min       2.089796
25%       2.910127
50%       4.225601
75%       6.640178
max      49.494253
Name: spike_ratio, dtype: float64
count    60.000000
mean      5.312970
std       6.384340
min       2.089796
25%       2.801587
50%       3.677025
75%       5.574731
max      49.494253
Name: spike_ratio, dtype: float64


In [13]:
top_stats = candidates[candidates["StockCode"].isin(top)].copy()
desc = df.groupby("StockCode")["Description"].agg(lambda s: s.mode().iat[0])
top_stats["description"] = top_stats["StockCode"].map(desc)

print(top_stats.sort_values("spike_ratio", ascending=False).head(5)
      [["StockCode", "description", "total_units", "max_week", "median_week", "spike_ratio"]])

code = top_stats.sort_values("spike_ratio", ascending=False).iloc[0]["StockCode"]
print(weekly[weekly["StockCode"] == code].sort_values("units", ascending=False).head(5))

returns = pd.read_parquet("../data/processed/returns.parquet")
print(returns[returns["StockCode"] == code].nsmallest(5, "Quantity")
      [["InvoiceNo", "Quantity", "InvoiceDate"]])

     StockCode                        description  total_units  max_week  \
1003     22086    PAPER CHAIN KIT 50'S CHRISTMAS         17158      2153   
410      21175        GIN + TONIC DIET METAL SIGN        11998      2127   
1029     22112         CHOCOLATE HOT WATER BOTTLE         5470       549   
1350     22470              HEART OF WICKER LARGE         9654      1368   
1591     22722  SET OF 6 SPICE TINS PANTRY DESIGN         5896      1036   

      median_week  spike_ratio  
1003         43.5    49.494253  
410         132.0    16.113636  
1029         40.0    13.725000  
1350        148.0     9.243243  
1591        116.0     8.931034  
      StockCode       week  units  units_regular  revenue  orders  avg_price
30119     22086 2011-11-14   2153         1585.0  9461.55     100   3.217429
30117     22086 2011-10-31   1745          845.0  5019.15      64   3.282308
30120     22086 2011-11-21   1691         1531.0  5698.21      96   3.199796
30121     22086 2011-11-28   1562    

In [14]:
all_weeks = pd.date_range(weekly["week"].min(), weekly["week"].max(), freq="W-MON")
idx = pd.MultiIndex.from_product([top, all_weeks], names=["StockCode", "week"])

panel = (weekly.set_index(["StockCode", "week"])
               .reindex(idx)
               .reset_index())

panel[["units", "units_regular", "revenue", "orders"]] = \
    panel[["units", "units_regular", "revenue", "orders"]].fillna(0)

panel["avg_price"] = panel.groupby("StockCode")["avg_price"].ffill().bfill()
print(panel.shape)

(3120, 7)


In [15]:
panel["bulk_units"] = panel["units"] - panel["units_regular"]
print("Total units removed by capping:",
      round(panel["bulk_units"].sum() / panel["units"].sum() * 100, 1), "%")
print(panel.nlargest(10, "bulk_units")[["StockCode", "week", "units", "units_regular", "bulk_units"]])

Total units removed by capping: 10.0 %
     StockCode       week   units  units_regular  bulk_units
388      22197 2011-05-23  4562.0          862.0      3700.0
294      84879 2011-08-01  3800.0         1080.0      2720.0
5       85123A 2011-01-10  3328.0          688.0      2640.0
2416     21977 2011-05-23  2825.0          605.0      2220.0
2917     21175 2011-01-10  2127.0          230.2      1896.8
2931     21175 2011-04-18  2098.0          201.2      1896.8
19      85123A 2011-04-18  2422.0          642.0      1780.0
410      22197 2011-10-24  2771.0         1379.0      1392.0
412      22197 2011-11-07  3900.0         2580.0      1320.0
398      22197 2011-08-01  2178.0          878.0      1300.0


In [16]:
print(panel["units_regular"].describe())
print(panel.nlargest(10, "units_regular")[["StockCode", "week", "units_regular"]])

count    3120.000000
mean      245.279997
std       236.829894
min         0.000000
25%       100.000000
50%       182.000000
75%       316.000000
max      2871.000000
Name: units_regular, dtype: float64
     StockCode       week  units_regular
415      22197 2011-11-28        2871.00
413      22197 2011-11-14        2612.00
412      22197 2011-11-07        2580.00
414      22197 2011-11-21        2555.00
95      85099B 2011-10-03        1822.42
399      22197 2011-08-08        1735.00
93      85099B 2011-09-19        1667.00
1297     22086 2011-11-14        1585.00
1299     22086 2011-11-28        1562.00
1298     22086 2011-11-21        1531.00


In [17]:
panel.to_parquet("../data/processed/weekly_demand.parquet", index=False)

## Weekly demand table
- Aggregated to weekly units per product (weeks start Monday), 52 full weeks.
- Kept products sold in at least 40 of 52 weeks (1,024 products), then took the top 60 by number of orders. Ranking by orders instead of units avoids bulk orders dominating the selection (overlap with the units ranking: 38 of 60).
- Filled missing weeks with zero demand: 60 products x 52 weeks = 3,120 rows.

## Bulk orders
- Single order lines of 1,900-4,300 units made up over 90% of some product-weeks (for example StockCode 22197, week of 23 May 2011). These are real but unpredictable wholesale purchases.
- Order lines are capped per product (maximum of the 99th percentile and 10x the median line size). Capping removes 10.0% of total units.
- Both `units` (actual) and `units_regular` (capped) are kept. Models forecast `units_regular`; bulk demand will be covered by safety stock in the optimizer.

## Seasonality
- The largest spike (StockCode 22086, Christmas paper chain kit, 2,153 units in the week of 14 Nov 2011 across 100 orders) is genuine seasonal demand and is kept.
